[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/03_advanced_architectures/vision_transformers.ipynb)


# 02 . Vision Transformers (ViT)

## Concept — Vision Transformer (revision notes)

**What it is**
- ViT cuts an image into fixed-size patches, linearly embeds each patch into a token, adds position embeddings and a learnable [CLS] token, and feeds the sequence to a standard Transformer encoder.
- The final [CLS] vector goes through a linear head for classification.

**Why it matters**
- It shows that a plain Transformer, with almost no image-specific machinery, can do vision when given enough data.
- It unifies vision and language architectures, enabling shared tooling and multimodal models.

**When to use**
- Large-data regimes (or with pretraining/strong augmentation) where learning spatial structure from data pays off.
- On small datasets from scratch, CNNs usually win because convolution bakes in locality and translation equivariance (see the experiment below).

**Math & intuition**
- An `H x W` image with patch size `P` gives `N = HW / P^2` tokens; each is a flattened `P*P*C` vector projected to `d_model`.
- Patch embedding is exactly a `Conv2d(C, d, kernel_size=P, stride=P)`.
- Self-attention cost is O(N^2), so smaller patches (more tokens) are quadratically more expensive.
- Position embeddings are essential: without them the model sees an unordered bag of patches.

### 1. Patchify an image by reshaping

**What this cell does (plain language):** we turn a (1, 3, 16, 16) image into a sequence of 16 flattened 4x4 patches using only `unfold`/`reshape`, then verify that patch 0 is the top-left corner and patch 1 is the block to its right (row-major order).

In [1]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
img = torch.randn(1, 3, 16, 16)
P = 4
# unfold windows of size PxP with stride P along H then W -> (B, C, H/P, W/P, P, P)
patches = img.unfold(2, P, P).unfold(3, P, P)
print("after unfold:", patches.shape)
patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(1, -1, 3 * P * P)   # (B, N, C*P*P): one flat vector per patch
print("patch sequence:", patches.shape, "-> N =", patches.shape[1], "tokens of dim", patches.shape[2])
print("patch 0 == top-left block:", torch.equal(patches[0, 0], img[0, :, :4, :4].reshape(-1)))
print("patch 1 == block to the right:", torch.equal(patches[0, 1], img[0, :, :4, 4:8].reshape(-1)))


after unfold: torch.Size([1, 3, 4, 4, 4, 4])
patch sequence: torch.Size([1, 16, 48]) -> N = 16 tokens of dim 48
patch 0 == top-left block: True
patch 1 == block to the right: True


### 2. Patch embedding = strided convolution

**What this cell does (plain language):** projecting every flattened patch with one shared Linear layer is mathematically identical to a Conv2d whose kernel and stride both equal the patch size. We copy the weights between the two and check the outputs match, then use the (faster, simpler) conv form.

In [2]:
d = 32
lin = nn.Linear(3 * P * P, d)
conv = nn.Conv2d(3, d, kernel_size=P, stride=P)
with torch.no_grad():
    conv.weight.copy_(lin.weight.view(d, 3, P, P))   # Linear weight (d, C*P*P) viewed as conv kernels (d, C, P, P)
    conv.bias.copy_(lin.bias)
a = lin(patches)                                     # (1, N, d)
b = conv(img).flatten(2).transpose(1, 2)             # (1, d, 4, 4) -> (1, N, d)
print("linear on patches vs conv:", a.shape, b.shape, "| equal:", torch.allclose(a, b, atol=1e-5))


linear on patches vs conv: torch.Size([1, 16, 32]) torch.Size([1, 16, 32]) | equal: True


### 3. Synthetic image dataset (no downloads)

**What this cell does (plain language):** we generate 16x16 grayscale images of four classes: horizontal bar, vertical bar, diagonal line, and a filled square. Each is placed at a random position with random brightness plus Gaussian noise, so the model must recognise shape regardless of location. Everything is generated on the fly with fixed seeds.

In [3]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
def make_image(c, g):
    im = torch.zeros(16, 16)
    r, col = torch.randint(2, 12, (2,), generator=g).tolist()
    if c == 0: im[r, 2:14] = 1                       # horizontal bar
    elif c == 1: im[2:14, col] = 1                   # vertical bar
    elif c == 2:                                     # diagonal
        o = torch.randint(0, 4, (1,), generator=g).item()
        for i in range(10): im[i + o, i + o] = 1
    else: im[r:r + 4, col:col + 4] = 1               # square
    return im * (0.6 + 0.4 * torch.rand(1, generator=g)) + 0.15 * torch.randn(16, 16, generator=g)

def make_dataset(n, seed):
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, 4, (n,), generator=g)
    x = torch.stack([make_image(int(c), g) for c in y]).unsqueeze(1)    # (n,1,16,16)
    return x, y
Xtr, ytr = make_dataset(1200, 0); Xte, yte = make_dataset(400, 1)
print("train:", Xtr.shape, "test:", Xte.shape, "| class counts:", torch.bincount(ytr).tolist())
fig, axs = plt.subplots(1, 4, figsize=(8, 2.2))
for c, ax in enumerate(axs):
    ax.imshow(Xtr[(ytr == c).nonzero()[0, 0], 0], cmap="gray"); ax.set_title(["hbar", "vbar", "diag", "square"][c]); ax.axis("off")
plt.tight_layout(); plt.show()


train: torch.Size([1200, 1, 16, 16]) test: torch.Size([400, 1, 16, 16]) | class counts: [306, 300, 294, 300]


### 4. The ViT model

**What this cell does (plain language):** we assemble the full model: conv patch-embed, prepend a learnable [CLS] token, add learnable position embeddings, run a stack of pre-LN Transformer encoder layers (PyTorch's `nn.TransformerEncoderLayer` with `norm_first=True`), and classify from the [CLS] output.

In [4]:
class ViT(nn.Module):
    def __init__(self, img=16, patch=4, in_ch=1, d=48, depth=2, heads=4, n_cls=4, use_pos=True):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_ch, d, patch, stride=patch)
        n = (img // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, d))                      # learnable summary token
        self.pos = nn.Parameter(torch.randn(1, n + 1, d) * 0.02) if use_pos else None
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm, self.head = nn.LayerNorm(d), nn.Linear(d, n_cls)
    def forward(self, x):
        t = self.patch_embed(x).flatten(2).transpose(1, 2)                 # (B,N,d)
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1)         # prepend CLS -> (B,N+1,d)
        if self.pos is not None: t = t + self.pos
        return self.head(self.norm(self.encoder(t)[:, 0]))                 # classify from CLS

torch.manual_seed(0)
vit = ViT()
print("logits:", vit(Xtr[:5]).shape, "| params:", sum(p.numel() for p in vit.parameters()))


logits: torch.Size([5, 4]) | params: 58516


### 5. Train the ViT

**What this cell does (plain language):** we train with AdamW and cross-entropy for a few epochs on the synthetic data, printing test accuracy. Chance is 25%, so anything far above that means the Transformer has learned shape-from-patches.

In [5]:
import time
def train(model, epochs=8, lr=2e-3, bs=64, seed=0, data=None, verbose=True):
    Xtr, ytr, Xte, yte = data if data is not None else (globals()["Xtr"], globals()["ytr"], globals()["Xte"], globals()["yte"])
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): acc = (model(Xte).argmax(1) == yte).float().mean().item()
        if verbose and (ep % 2 == 1 or ep == epochs - 1): print(f"epoch {ep+1} | last batch loss {loss.item():.3f} | test acc {acc:.3f}")
    return acc
t0 = time.time(); torch.manual_seed(0); vit = ViT(); acc_vit = train(vit); print(f"time {time.time()-t0:.1f}s")


epoch 2 | last batch loss 0.525 | test acc 0.812


epoch 4 | last batch loss 0.100 | test acc 0.985


epoch 6 | last batch loss 0.010 | test acc 0.995


epoch 8 | last batch loss 0.011 | test acc 0.988
time 9.5s


### 6. Do position embeddings matter? Shuffle the patches

**What this cell does (plain language):** after training we randomly permute the patch tokens at test time and compare accuracy on normal vs shuffled patches. On this easy task the damage may be modest, because a bar's patch CONTENT (a horizontal vs vertical stroke inside the patch) already reveals its class; the next cell uses a task where layout is everything.

In [6]:
@torch.no_grad()
def acc_shuffled(model, X, y, seed=0):
    g = torch.Generator().manual_seed(seed)
    t = model.patch_embed(X).flatten(2).transpose(1, 2)               # tokens (B,N,d)
    perm = torch.randperm(t.size(1), generator=g)
    t = t[:, perm]                                                    # scramble spatial layout
    t = torch.cat([model.cls.expand(len(t), -1, -1), t], 1) + model.pos
    return (model.head(model.norm(model.encoder(t)[:, 0])).argmax(1) == y).float().mean().item()
vit.eval()
print(f"normal test acc: {acc_vit:.3f} | patches shuffled: {acc_shuffled(vit, Xte, yte):.3f}")


normal test acc: 0.988 | patches shuffled: 0.952


### 7. Ablation: remove position embeddings on a layout-dependent task

**What this cell does (plain language):** the shapes task can be solved from patch contents alone, so we build a second task where only WHERE things are matters: a single bright 4x4 blob in one of the four quadrants (label = quadrant). Every patch looks the same regardless of label except for its location in the grid, so a bag of patches has nothing to go on. We train with and without position embeddings and print both accuracies; chance is 25%.

In [7]:
def make_quadrants(n, seed):
    g = torch.Generator().manual_seed(seed); y = torch.randint(0, 4, (n,), generator=g); x = 0.1 * torch.randn(n, 1, 16, 16, generator=g)
    for i, c in enumerate(y.tolist()):
        r0 = (c // 2) * 8 + int(torch.randint(0, 5, (1,), generator=g)); c0 = (c % 2) * 8 + int(torch.randint(0, 5, (1,), generator=g))
        x[i, 0, r0:r0 + 4, c0:c0 + 4] += 1.0
    return x, y
Xq, yq = make_quadrants(1200, 5); Xqt, yqt = make_quadrants(400, 6)
qdata = (Xq, yq, Xqt, yqt)
t0 = time.time()
torch.manual_seed(0); acc_q_pos = train(ViT(use_pos=True), epochs=8, data=qdata, verbose=False)
torch.manual_seed(0); acc_q_nopos = train(ViT(use_pos=False), epochs=8, data=qdata, verbose=False)
print(f"quadrant task | with position embeddings: {acc_q_pos:.3f} | without: {acc_q_nopos:.3f} | {time.time()-t0:.1f}s")


quadrant task | with position embeddings: 1.000 | without: 0.225 | 11.6s


### 8. Attention from [CLS] to patches

**What this cell does (plain language):** we compute the first layer's attention of the [CLS] token over the 16 patches for one test image of each class and show it as a 4x4 heatmap beside the image. Early-layer attention is often diffuse; the point is how to extract and reshape it back to the patch grid.

In [8]:
@torch.no_grad()
def cls_attention(model, x):
    t = model.patch_embed(x).flatten(2).transpose(1, 2)
    t = torch.cat([model.cls.expand(len(t), -1, -1), t], 1) + model.pos
    layer = model.encoder.layers[0]
    h = layer.norm1(t)                                                   # norm_first=True: attention sees normed input
    _, w = layer.self_attn(h, h, h, need_weights=True, average_attn_weights=True)   # (B, N+1, N+1) averaged over heads
    return w[:, 0, 1:].reshape(len(x), 4, 4)                             # CLS row, drop CLS column -> patch grid
vit.eval()
fig, axs = plt.subplots(2, 4, figsize=(8, 4))
for c in range(4):
    x = Xte[(yte == c).nonzero()[0, 0]][None]
    axs[0, c].imshow(x[0, 0], cmap="gray"); axs[0, c].axis("off")
    axs[1, c].imshow(cls_attention(vit, x)[0], cmap="magma"); axs[1, c].axis("off")
axs[0, 0].set_title("image"); axs[1, 0].set_title("CLS attention (4x4)")
plt.tight_layout(); plt.show()


### 9. Patch size vs compute

**What this cell does (plain language):** halving the patch size quadruples the number of tokens and therefore ~16x the attention matrix. We count tokens, attention-matrix entries and wall time of a forward pass for patch sizes 8, 4 and 2 on the same 16x16 images.

In [9]:
for p in [8, 4, 2]:
    m = ViT(patch=p).eval(); n = (16 // p) ** 2 + 1
    t0 = time.time()
    with torch.no_grad(): m(Xte[:200])
    print(f"patch {p}: tokens={n:3d} | attention entries/head={n*n:6d} | params={sum(q.numel() for q in m.parameters()):,} | fwd {1000*(time.time()-t0):.0f} ms")


patch 8: tokens=  5 | attention entries/head=    25 | params=60,244 | fwd 10 ms


patch 4: tokens= 17 | attention entries/head=   289 | params=58,516 | fwd 29 ms


patch 2: tokens= 65 | attention entries/head=  4225 | params=60,244 | fwd 165 ms


## Common bugs

- **Forgetting the [CLS] token's position slot** — position table sized `N` instead of `N+1` gives a shape mismatch. Fix: size it `N + 1` when you prepend CLS.
- **Image size not divisible by patch size** — `Conv2d` silently drops the remainder pixels. Fix: assert `H % P == 0 and W % P == 0` or pad/resize.
- **Reshaping patches in the wrong order** — `reshape` straight from (B,C,H,W) interleaves pixels from different patches, so tokens are garbage. Fix: use unfold + permute, or the strided conv.
- **Fixed position embeddings with a new image size** — a 224-trained table cannot index a larger grid. Fix: interpolate the position embeddings (2D bicubic) when changing resolution.
- **Expecting ViT to beat a CNN on tiny data** — without inductive bias it needs lots of data or heavy augmentation/pretraining. Fix: use a CNN, hybrid stem, or pretrained ViT for small datasets.
- **`norm_first` mismatch when reading attention** — with `norm_first=True` the attention sees `norm1(x)`, so hooking raw `x` gives different maps (section 8 normalises first).
- **Wrong batch layout for `nn.TransformerEncoder`** — default is (T,B,D). Fix: `batch_first=True` as done here.
- **Not normalising the images** — unnormalised inputs make patch embeddings poorly scaled and slow training. Fix: standardise per channel.

## Exercises

**Exercise 1.** Change the classifier to mean-pool the patch tokens instead of using [CLS] (remove the CLS token) and compare test accuracy.

In [10]:
# Your attempt for Exercise 1 here


**Solution 1**

In [11]:
class ViTMean(ViT):
    def __init__(self):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, 16, 48) * 0.02)       # no CLS slot
    def forward(self, x):
        t = self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos
        return self.head(self.norm(self.encoder(t).mean(1)))
torch.manual_seed(0); m = ViTMean(); print("mean-pool acc:", round(train(m, epochs=6), 3))


epoch 2 | last batch loss 0.408 | test acc 0.910


epoch 4 | last batch loss 0.020 | test acc 0.988


epoch 6 | last batch loss 0.113 | test acc 0.998
mean-pool acc: 0.998


**Exercise 2.** Verify that the number of tokens for a 32x32 image with patch 8 is 16 (+1 CLS) and print the patch-embed output shape.

In [12]:
# Your attempt for Exercise 2 here


**Solution 2**

In [13]:
pe = nn.Conv2d(1, 48, 8, stride=8)
out = pe(torch.randn(2, 1, 32, 32)).flatten(2).transpose(1, 2)
print(out.shape, "-> tokens", out.shape[1], "(+1 CLS =", out.shape[1] + 1, ")")


torch.Size([2, 16, 48]) -> tokens 16 (+1 CLS = 17 )


**Exercise 3.** Add random translation augmentation (roll the image by up to 2 pixels) to the training loop and see if test accuracy changes.

In [14]:
# Your attempt for Exercise 3 here


**Solution 3**

In [15]:
def train_aug(model, epochs=6, bs=64):
    torch.manual_seed(0); opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]; xb = Xtr[idx]
            dx, dy = torch.randint(-2, 3, (2,)).tolist()
            xb = torch.roll(xb, shifts=(dy, dx), dims=(2, 3))           # one random shift per batch
            loss = F.cross_entropy(model(xb), ytr[idx]); opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad(): return (model(Xte).argmax(1) == yte).float().mean().item()
torch.manual_seed(0); print("aug acc:", round(train_aug(ViT()), 3))


aug acc: 0.995


**Exercise 4.** Compute per-class test accuracy of the trained `vit`.

In [16]:
# Your attempt for Exercise 4 here


**Solution 4**

In [17]:
with torch.no_grad(): pred = vit(Xte).argmax(1)
for c, name in enumerate(["hbar", "vbar", "diag", "square"]):
    m = yte == c; print(name, round((pred[m] == c).float().mean().item(), 3))


hbar 1.0
vbar 0.948
diag 1.0
square 1.0


**Exercise 5.** Write a tiny CNN (2 conv layers + pooling + linear) and compare its parameter count and test accuracy with the ViT after the same number of epochs.

In [18]:
# Your attempt for Exercise 5 here


**Solution 5**

In [19]:
cnn = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 4))
torch.manual_seed(0)
print("CNN params:", sum(p.numel() for p in cnn.parameters()), "vs ViT:", sum(p.numel() for p in vit.parameters()))
print("CNN acc:", round(train(cnn), 3))


CNN params: 1316 vs ViT: 58516


epoch 2 | last batch loss 1.354 | test acc 0.243


epoch 4 | last batch loss 1.223 | test acc 0.830


epoch 6 | last batch loss 0.905 | test acc 0.962


epoch 8 | last batch loss 0.551 | test acc 0.993
CNN acc: 0.993
